In [145]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import math

In [146]:
FILE = 'output.csv'

"""
raw df has continous time series data for each user, no gap between days.
1 sugg.select.utime is Nan: invalid
Number of unique users: 37
"""
df = pd.read_csv(FILE)


In [147]:
cols_dropped = ['sugg.select.utime', 'sugg.response.utime', 'dec.location.category', 'decision.index.nogap']

sdf = df.drop(columns=cols_dropped)

print(sdf.columns)

cols = ['uid', 'datetime', 'date', 'day_slot', 'is_randomized', 'avail', 'send', 'send_active', 'send_sedentary', 'returned_message', 'response', 'activity', 'location', 'weather', 'temperature', 'jbsteps30', 'jbsteps30pre']



sdf.columns = cols

print(sdf.columns)

Index(['user.index', 'datetime', 'date', 'sugg.select.slot', 'is.randomized',
       'avail', 'send', 'send.active', 'send.sedentary', 'returned.message',
       'response', 'recognized.activity', 'location_group',
       'dec.weather.condition', 'dec.temperature', 'jbsteps30',
       'jbsteps30pre'],
      dtype='str')
Index(['uid', 'datetime', 'date', 'day_slot', 'is_randomized', 'avail', 'send',
       'send_active', 'send_sedentary', 'returned_message', 'response',
       'activity', 'location', 'weather', 'temperature', 'jbsteps30',
       'jbsteps30pre'],
      dtype='str')


In [148]:
sdf.isnull().sum()
sdf[sdf['temperature'].isnull()].groupby(['uid', 'datetime']).size()

uid  datetime           
2    2015-08-02 22:00:00    1
3    2015-08-06 20:30:00    1
6    2015-08-17 16:00:00    1
7    2015-08-21 22:00:00    1
     2015-08-21 23:30:00    1
                           ..
33   2016-01-01 16:00:00    1
     2016-01-13 14:05:00    1
35   2015-12-15 12:00:00    1
37   2015-12-15 12:00:00    1
     2016-01-13 22:25:00    1
Length: 105, dtype: int64

In [149]:


# df.fillna()
# df.median()
# df.interpolate()

sdf['datetime'] = pd.to_datetime(sdf['datetime'])
sdf = sdf.set_index('datetime')

sdf['temperature'] = sdf.groupby('uid')['temperature'].transform(
    lambda x: x.interpolate(method='time').ffill().bfill()
)

# 3. 恢复索引
sdf = sdf.reset_index()

sdf[sdf['temperature'].isnull()].groupby(['uid', 'datetime']).size()

Series([], dtype: int64)

In [150]:
# 先把 unknown 和报错字符串替换为 NaN
sdf['weather'] = sdf['weather'].replace(
    ['unknown',
     'com.google.appengine.labs.repackaged.org.json.JSONObject.getJSONObject(JSONObject.java:516)',
     'com.google.appengine.labs.repackaged.org.json.JSONObject.<init>(JSONObject.java:179)'],
    pd.NA
)


sdf[sdf['weather'].isnull()].groupby(['uid', 'datetime']).size()

# 用同一用户同一天的众数填补
def fill_mode(x):
    mode = x.mode()
    return x.fillna(mode[0] if len(mode) > 0 else pd.NA)

sdf['weather'] = sdf.groupby(['uid', 'date'])['weather'].transform(fill_mode)


# 剩余的用前向填充
sdf['weather'] = sdf.groupby('uid')['weather'].ffill()

sdf[sdf['weather'].isnull()].groupby(['uid', 'datetime']).size()



sdf.isnull().sum()

datetime               0
uid                    0
date                   0
day_slot               0
is_randomized          0
avail                  0
send                   0
send_active            0
send_sedentary         0
returned_message       0
response            4017
activity               0
location               0
weather                0
temperature            0
jbsteps30              0
jbsteps30pre           0
dtype: int64

In [151]:
'''
send:
 - 0: no send
 - 1: active
 - 2: sedentary

when send.active == False and send.sedentary == False, returned.message is always 'donotnotify'
when send == False, returned.message is always 'donotnotify'
when returned.message == 'donotnotify', send == False
'''
# sdf[sdf['send'] == 0][['send_active', 'send_sedentary']].value_counts()
# sdf[sdf['send_active'] == 0 & (sdf['send_sedentary'] == 0)]['send'].value_counts()
sdf['send'] = sdf['send_active'].astype(int) + sdf['send_sedentary'].astype(int) * 2
sdf['send'].value_counts()

sdf.drop(columns=['send_active', 'send_sedentary'], inplace=True)

sdf.columns

Index(['datetime', 'uid', 'date', 'day_slot', 'is_randomized', 'avail', 'send',
       'returned_message', 'response', 'activity', 'location', 'weather',
       'temperature', 'jbsteps30', 'jbsteps30pre'],
      dtype='str')

In [152]:
sdf[sdf['response'].isnull()]['send'].value_counts()
# sdf[sdf['response'].isnull()]['returned_message'].value_counts()


sdf.loc[sdf['response'].isnull() & (sdf['send'] == 0), 'response'] = 'no_send'
sdf[sdf['response'].isnull()]['send'].value_counts()

sdf.loc[sdf['response'].isnull() & (sdf['send'] != 0), 'response'] = 'no_response'

In [153]:
# print(sdf.isnull().sum())

sdf['date'] = pd.to_datetime(sdf['date'])

sdf['study_day'] = (sdf['date'] - sdf.groupby('uid')['date'].transform('min')).dt.days + 1

# print(sdf.head(5).to_string())

print(sdf.isnull().sum())

# sdf.to_csv('./cleaned_output.csv', index=False)

datetime            0
uid                 0
date                0
day_slot            0
is_randomized       0
avail               0
send                0
returned_message    0
response            0
activity            0
location            0
weather             0
temperature         0
jbsteps30           0
jbsteps30pre        0
study_day           0
dtype: int64


In [154]:
sdf = sdf.loc[~(~sdf['avail'] & (sdf['send'] > 0))]
sdf.loc[sdf['activity'] == 'UNKNOWN', 'activity'] = 'STILL'
sdf.loc[sdf['activity'] != 'STILL', 'activity'] = 'ON_MOVE'

sdf.loc[(sdf['response'] == 'snoozed_for_4_hours') | (sdf['response'] == 'snoozed_for_12_hours'), 'response'] = 'no_response'


In [155]:
sdf['is_weekday'] = sdf['date'].dt.weekday < 5


In [156]:
location_mapping = {
    # 高活动
    'Parks & Recreation - Outdoor High-Activity': 'Park',
    'Sports & Fitness - Indoor High-Activity': 'Gym',
    # 室外低活动
    'Auto & Transport - Outdoor Low-Activity': 'Transit',

    # 固定地点
    'Home': 'Home',
    'Work': 'Work',
    'University/School': 'University',
    'Food & Dining': 'Restaurant',
    'Healthcare & Personal Care - Indoor Low-Activity': 'Healthcare',

    # 商店类（含 Leisure）
    'Just Store': 'Store',
    'Other Specialty Store': 'Store',
    'Leisure & Entertainment- Indoor Low-Activity': 'Store',  # 改：9 条合并到 Store
    'Electronics Store': 'ElectronicsStore',
    'Home & Furniture Store': 'FurnitureStore',
    'Clothing & Fashion Store': 'ClothingStore',
    'Grocery & Convenience Store': 'GroceryStore',
    # 兜底
    'Unknown': 'Unknown',
}

weather_mapping = {
    # 晴朗
    'Clear': 'Clear',

    # 多云家族
    'Partly Cloudy': 'PartlyCloudy',
    'Scattered Clouds': 'PartlyCloudy',
    'Mostly Cloudy': 'MostlyCloudy',
    'Overcast': 'Overcast',

    # 降雨家族
    'Rain': 'Rain',
    'Light Rain': 'Rain',
    'Thunderstorm': 'Rain',
    'Light Thunderstorms and Rain': 'Rain',
    'Light Freezing Rain': 'Rain',

    # 降雪家族
    'Snow': 'Snow',
    'Light Snow': 'Snow',
    'Ice Pellets': 'Snow',

    # 低能见度
    'Fog': 'Fog',
    'Haze': 'Fog',
    'Light Freezing Fog': 'Fog',
}

# print(sdf['location'].value_counts())

sdf['location'] = sdf['location'].map(location_mapping)

# print(sdf['location'].value_counts())
sdf['weather'] = sdf['weather'].map(weather_mapping)

# 安全检查（强烈建议加）
assert sdf['location'].notna().all(), "有未映射的 location！"
assert sdf['weather'].notna().all(), "有未映射的 weather！"

# sdf.to_csv('./cleaned_output.csv', index=False)





In [160]:
def bucket_temperature(t):
    """温度分桶（基于体感）"""
    if t < -5:    return 'temp_freezing'    # 极冷, n=277
    elif t < 5:   return 'temp_cold'        # 冷, n=1323
    elif t < 15:  return 'temp_cool'        # 凉爽, n=2133
    elif t < 22:  return 'temp_mild'        # 温和, n=1266
    elif t < 28:  return 'temp_warm'        # 温暖, n=1280
    else:         return 'temp_hot'         # 炎热, n=446

sdf['temperature'] = sdf['temperature'].apply(bucket_temperature)

In [161]:

sdf.to_csv('./cleaned_output.csv', index=False)
sdf.to_csv('../data/cleaned_output.csv', index=False)